<a href="https://colab.research.google.com/github/YokoyamaLab/PythonBasics/blob/2025/28_day08kk_YoloPose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 提出者情報
No = "G000000" # @param {"type":"string","placeholder":"学籍番号"}
名前 = "" # @param {"type":"string","placeholder":"名前"}


In [ ]:
# 必要なライブラリのインストール
!pip install opencv-python tensorflow ipywidgets ultralytics sqids

import cv2
import numpy as np
from google.colab.patches import cv2_imshow
import ipywidgets as widgets
from IPython.display import display
from ultralytics import YOLO
from IPython.display import clear_output
from PIL import Image, ImageDraw, ImageFont
from sqids import Sqids
import datetime
import sys
import shutil
import os

current_execution = "";

In [ ]:
# 体の以下のパーツの位置を認識可能です
parts = [
    'NOSE',
    "LEFT_EYE",
    "RIGHT_EYE",
    "LEFT_EAR",
    "RIGHT_EAR",
    "LEFT_SHOULDER",
    "RIGHT_SHOULDER",
    "LEFT_ELBOW",
    "RIGHT_ELBOW",
    "LEFT_WRIST",
    "RIGHT_WRIST",
    "LEFT_HIP",
    "RIGHT_HIP",
    "LEFT_KNEE",
    "RIGHT_KNEE",
    "LEFT_ANKLE",
    "RIGHT_ANKLE"
]

In [ ]:

if 'No' not in locals() or No == "":
  print("ノート冒頭で学籍番号を入力してください。")
  sys.exit()

# YOLOv11モデルのロード
model = YOLO('yolo11x-pose.pt')

# 画像のファイル名を読み込み、体のパーツの位置(キーポイント)を返す
def detect_pose(image_path):
    # Read the image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not read image from {image_path}")
        return None, None

    # ポーズ推定を行う
    results = model(img)

    # キーポイント記録用・結果画像
    keypoints_list = []
    annotated_img = img.copy()

    for r in results:
        if r.keypoints is not None:
            kps = r.keypoints.xy.cpu().numpy()
            for person_kps in kps:
                keypoints_list.append(person_kps.tolist())
            annotated_img = r.plot()
    return keypoints_list, annotated_img, img

# ファイルアップロードウィジェットの初期化
uploader = widgets.FileUpload(
    accept='.jpg,.heic',  # Accepted file extensions
    multiple=False  # Allow only single file upload
)

# アップロードボタンの表示
print("Please upload a JPG image for pose estimation:")
display(uploader)

# アップローダー本体（アップロード後ポーズ推定を呼び出し結果表示）
def handle_upload(change):
    # 実行事にユニークなファイル名を生成する
    dt = datetime.datetime.now()
    sqids = Sqids(min_length=10)
    current_execution = sqids.encode([dt.hour, dt.minute, dt.second])
    filename_original = f"{current_execution}-original.jpg"
    filename_annotated = f"{current_execution}-annotated.jpg"
    filename_result = f"{current_execution}-result.jpg"
    if uploader.value:
        uploaded_file_name = list(uploader.value.keys())[0]
        uploaded_file_content = uploader.value[uploaded_file_name]['content']

        with open(filename_original, 'wb') as f:
            f.write(uploaded_file_content)

        clear_output()
        print(f"Uploaded file: {filename_original}")

        # 上記detect_pose関数呼び出し（返り値はキーポイント、推定結果画像、元画像）
        keypoints, annotated_image, image = detect_pose(filename_original)

        if keypoints is not None:
            # 認識した人物毎のキーポイントを出力
            print("\nDetected Keypoints (List of lists):")
            for i, kps in enumerate(keypoints):
                print(f"Person {i+1}: {kps}")

            if annotated_image is not None:
                print("\nAnnotated Image:")
                #物体認識・ポーズ推定結果画像の表示と保存
                cv2_imshow(annotated_image)
                cv2.imwrite(filename_annotated, annotated_image)

                # 元画像に対してキーポイントに基づいてPILで描画する
                pilimg = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
                draw = ImageDraw.Draw(pilimg)

                # 【課題】元画像の上にキーポイント情報を利用してお絵描きしてください
                #     (例えば目隠し線を入れる、友達同士の手と手を赤い糸で結ぶ等)

                # とりあえず例として全てのキーポイント位置上にキーポイントの名前を表示＆左目に赤丸
                for i, kps in enumerate(keypoints):
                  for p, kp in enumerate(kps):
                    # kp: 画像中の座標
                    # parts[p]: キーポイントの名称(上記partsリスト参照の事)

                    # 1pxずらして黒を書いた上に白を描く、影付き文字の古典的テクニック
                    draw.text(list(map(lambda x: x + 1, kp)),parts[p], 'black')
                    draw.text(list(map(lambda x: x - 1, kp)),parts[p], 'black')
                    draw.text(kp,parts[p], 'white')

                    # 左目を見つけたら赤い●を描く
                    if(parts[p] == "LEFT_EYE"):
                      draw.circle(kp, 5, fill='red')
                # PIL描画ここまで

                # PIL描画結果の表示と保存
                display(pilimg)
                pilimg.save(filename_result, 'JPEG')
        else:
            print("No keypoints detected or error reading image.")


# ファイルが指定されたらhandle_upload関数を呼び出す
uploader.observe(handle_upload, names='value')


画像提出のコードです。

In [ ]:
# 画像保存のためのコード
from google.colab import drive
drive.mount("/content/gdrive")
our_dir = "/content/gdrive/Shareddrives/2025-LG080A01／情報科学 c/yolo/"

def copy_file_to_our_dir(filename):
  if not os.path.exists(our_dir):
      os.makedirs(our_dir)
  shutil.copy2(filename, our_dir)
  print(f"'{filename}' を '{our_dir}' にコピーしました。")

copy_file_to_our_dir(filename_original)
copy_file_to_our_dir(filename_annotated)
copy_file_to_our_dir(filename_result)